In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

workspace_root = Path.cwd().parent
env_paths = (
    Path.cwd() / ".env",
    workspace_root / ".env",
    workspace_root / "Langchain_Basics" / ".env",
)

for env_path in env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"Loaded environment from: {env_path}")
        break
else:
    print("No .env file found. Create Agents/.env or workspace-root/.env.")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "LangChainTrainings-Agents")

if os.getenv("LANGSMITH_API_KEY"):
    print(f"LangSmith tracing enabled for project: {os.environ['LANGSMITH_PROJECT']}")
else:
    print("Add LANGSMITH_API_KEY to .env to enable LangSmith tracing.")

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2500,
    reasoning=False,
)

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# 1. Documents
docs = [
    Document(page_content="This is a test document for RAG testing."),
    Document(page_content="plawright is more powerful than playwright."),
    Document(page_content="Selenium is a powerful tool for web automation."),
    Document(page_content="Appium is a popular framework for mobile app testing."),
]

references = [
    "Selenium is a powerful tool for web automation.",
    "Appium is a popular framework for mobile app testing.",
    "Playwright is a tool used for web automation."
]

questions = [
    "What is Selenium used for?",
    "What is Appium used for?",
    "What is Playwright used for?",
]

# 2. Create embedding model
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 3. Create vector store
vector_store = Chroma.from_documents(documents=docs,embedding=embeddings)

# 4. Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 5. Retrieve relevant documents
question = "What is Selenium used for?"

results = retriever.invoke(question)
results

results
# 6. Display retrieved documents
# for doc in results:
#     print(doc.page_content)
 
dataset = []

for question, reference in zip(questions, references):
    # Retrieve relevant documents
    results = retriever.invoke(question)

    # Convert retrieved Document objects to strings
    relevant_docs = []
    for doc in results:
        relevant_docs.append(doc.page_content)

    # Generate the answer from the retrieved context
    response = llm.invoke(
        "Answer the question using only the provided context.\n\n"
        f"Context:\n{chr(10).join(relevant_docs)}\n\n"
        f"Question: {question}\nAnswer:"
    ).content

    dataset.append({
        "user_input": question,
        "retrieved_contexts": relevant_docs,
        "response": response,
        "reference": reference
    })

dataset   
    

In [ ]:
# Evaluation of datasets

from ragas import EvaluationDataset, evaluate
from ragas.metrics import ContextRecall, Faithfulness
from ragas.llms import LangchainLLMWrapper

evaluation_dataset = EvaluationDataset.from_list(dataset)

evaluation_llm = LangchainLLMWrapper(llm)

result = evaluate(
    evaluation_dataset,
    metrics=[ContextRecall(), Faithfulness()],
    llm=evaluation_llm,
)

result.to_pandas